In [1]:
pip install python-docx

In [ ]:
import json
import os
import requests
import docx

#Read the document file
def read_word_file(file_path):
    doc = docx.Document(file_path)
    full_text = []
    for paragraph in doc.paragraphs:
        full_text.append(paragraph.text)
    return '\n'.join(full_text)


# 1. This remains a raw DICT, but booleans in JSON (False/True) must be Python tokens (False/True)
def generate_expected_json_template():
    template = {
        "type": "object",
        "additionalProperties": False,
        "required": [
            "stage1_ingestion",
            "catalyst",
            "actor_characteristics",
            "attack_characteristics",
            "organisation_characteristics",
        ],
        "properties": {
            "stage1_ingestion": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "case_id",
                    "source",
                    "reference",
                    "case_summary",
                    "include_for_stage_2",
                    "screening_note",
                ],
                "properties": {
                    "case_id": {
                        "type": "string",
                        "pattern": "^CS-[0-9]{3,}$",
                        "description": "Unique case identifier (e.g., CS-001).",
                    },
                    "source": {
                        "type": "string",
                        "description": "Source of the case data.",
                    },
                    "reference": {
                        "type": "string",
                        "description": "Unique case reference, citation, URL, or document name.",
                    },
                    "case_summary": {
                        "type": "string",
                        "description": "Short factual summary of the case.",
                    },
                    "include_for_stage_2": {
                        "type": "boolean",
                        "description": "Whether the case proceeds to Stage 2 analysis.",
                    },
                    "screening_note": {
                        "type": "string",
                        "description": "Analyst screening notes.",
                    },
                },
            },
            "catalyst": {
                "type": "object",
                "additionalProperties": False,
                "required": ["precipitating_event"],
                "properties": {
                    "precipitating_event": {
                        "type": "array",
                        "description": "Triggering event or catalyst associated with the insider behaviour.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "employee_dismissal",
                                "demotion",
                                "resignation",
                                "disputes_with_employers",
                                "perceived_injustices",
                                "negative_company_acts",
                                "family_problems",
                                "coercion",
                                "new_opportunities",
                                "other",
                                "unknown",
                            ],
                        },
                    }
                },
            },
            "actor_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "actors",
                    "psychological_state",
                    "personality_characteristics",
                    "attitude_towards_work",
                    "motivation_to_attack",
                    "skill_set",
                    "opportunity",
                    "historical_behaviour",
                    "observed_physical_behaviour",
                    "observed_cyber_behaviour",
                ],
                "properties": {
                    "actors": {
                        "type": "array",
                        "description": "Actor-specific information. Add one object for each actor involved in the incident.",
                        "items": {
                            "type": "object",
                            "additionalProperties": False,
                            "required": [
                                "actor_id",
                                "type_of_actor",
                                "enterprise_role",
                                "state_of_relationship",
                            ],
                            "properties": {
                                "actor_id": {
                                    "type": "string",
                                    "description": "Identifier for the actor (e.g., actor_1, actor_2).",
                                },
                                "type_of_actor": {
                                    "type": "string",
                                    "enum": [
                                        "employee",
                                        "former_employee",
                                        "contractor",
                                        "business_partner",
                                        "other",
                                        "unknown",
                                    ],
                                },
                                "enterprise_role": {
                                    "type": "string",
                                    "enum": [
                                        "scientist",
                                        "engineer",
                                        "programmer",
                                        "salesperson",
                                        "manager",
                                        "security_person",
                                        "finance_person",
                                        "executive",
                                        "support_person",
                                        "military_person",
                                        "other",
                                        "unknown",
                                    ],
                                },
                                "state_of_relationship": {
                                    "type": "string",
                                    "enum": [
                                        "current",
                                        "former",
                                        "serving_notice",
                                        "temporary",
                                        "contract",
                                        "other",
                                        "unknown",
                                    ],
                                },
                            },
                        },
                    },
                    "psychological_state": {
                        "type": "array",
                        "description": "Actor's psychological or emotional state.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "financial_stress",
                                "grievances",
                                "misjudgement",
                                "rationalisation",
                                "disgruntlement",
                                "aggression",
                                "job_dissatisfaction",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "personality_characteristics": {
                        "type": "array",
                        "description": "Actor personality characteristics.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "greed",
                                "selfishness",
                                "personal_inflexibilities",
                                "narcissism",
                                "psychopathy",
                                "low_agreeableness",
                                "risk_tolerance",
                                "honesty_humility",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attitude_towards_work": {
                        "type": "array",
                        "description": "Actor's feelings or behavioural orientation toward work.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "dissatisfied",
                                "disengaged",
                                "resentful",
                                "hostile",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "motivation_to_attack": {
                        "type": "array",
                        "description": "Reason or motive for the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "financial_reward",
                                "personal_gain",
                                "revenge",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "skill_set": {
                        "type": "array",
                        "description": "Description of the actor's relevant skills or capabilities.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "technical_expertise",
                                "privileged_system_knowledge",
                                "data_exploitation_capability",
                                "deception/social_engineering_capability",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "opportunity": {
                        "type": "array",
                        "description": "Actor's opportunity to initiate or conduct the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "privileged_access",
                                "physical_access",
                                "remote_access",
                                "shared_credentials",
                                "no_access_revocation",
                                "insufficient_monitoring",
                                "trusted_position",
                                "third_party_access",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "historical_behaviour": {
                        "type": "array",
                        "description": "Relevant historical behaviour before the incident.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "addictive_practices",
                                "harassment",
                                "company_policy_violation",
                                "criminal_history",
                                "history_of_serious_mental_problems",
                                "carelessness",
                                "absent_mindedness",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "observed_physical_behaviour": {
                        "type": "array",
                        "description": "Observed physical-world behaviour relevant to the case.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "poor_work_performance",
                                "workplace_rule_violation",
                                "conflicts_with_colleagues",
                                "increased_outbursts",
                                "substance_abuse",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "observed_cyber_behaviour": {
                        "type": "array",
                        "description": "List of observed cyber activities.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "access_and_authentication",
                                "data_and_file_management",
                                "communication_activities",
                                "system_and_network_behaviour",
                                "privilege_and_security_behaviour",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
            "attack_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": [
                    "attack",
                    "attack_objective",
                    "attack_step",
                    "attack_step_goal",
                ],
                "properties": {
                    "attack": {
                        "type": "array",
                        "description": "Whether the harmful activity was intentional, unintentional, other, or unknown.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "intentional",
                                "unintentional",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_objective": {
                        "type": "array",
                        "description": "Primary objective or outcome associated with the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "data_theft",
                                "sabotage",
                                "fraud",
                                "espionage",
                                "mixed",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_step": {
                        "type": "array",
                        "description": "Specific activities undertaken to conduct the attack.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "preparation/planning",
                                "target_identification",
                                "information_gathering / Discovery",
                                "access/access_acquisition",
                                "privilege/permission_Abuse",
                                "collection/acquisition",
                                "execution/action",
                                "data_transfer/exfiltration",
                                "manipulation/modification",
                                "sabotage/disruption",
                                "concealment/evasion",
                                "persistence/continued_access",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "attack_step_goal": {
                        "type": "array",
                        "description": "Specific outcome/goal after the attack steps take place",
                        "items": {
                            "type": "string",
                            "enum": [
                                "confidentiality_loss",
                                "integrity_loss",
                                "availability_loss",
                                "financial_loss",
                                "reputational_damage",
                                "privacy_loss",
                                ":operational_disruption",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
            "organisation_characteristics": {
                "type": "object",
                "additionalProperties": False,
                "required": ["asset", "vulnerability"],
                "properties": {
                    "asset": {
                        "type": "array",
                        "description": "Targeted organisational asset, data, system, process, or service.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "information_asset",
                                "technology_asset",
                                "organisational_asset",
                                "intangible_asset",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                    "vulnerability": {
                        "type": "array",
                        "description": "Organisational or control vulnerabilities that enabled or worsened the incident.",
                        "items": {
                            "type": "string",
                            "enum": [
                                "lack_of_security_policies",
                                "inconsistent_security_policies",
                                "poor_security_practices",
                                "organisational_culture",
                                "lack_of_support",
                                "poor_communication",
                                "poor_management_practices",
                                "leadership_issues",
                                "insufficient_monitoring",
                                "lack_of_oversight",
                                "other",
                                "unknown",
                            ],
                        },
                    },
                },
            },
        },
    }
    return template

# Task:
# Analyse the following case study and extract information into the requested schema fields. Use the rules defined in the Coding Guide.
# When picking values for attack steps, conside MITRE Attack faremework https://attack.mitre.org/. Values in the framework should be exactly the same in the attack steps. Mention it with the tactic Id



# 2. Prompts can be more concise because the schema structure handles enforcement
def build_prompt(case_text, coding_guide_text):
    return f"""
### Role

You are an expert cybersecurity analyst specializing in insider threat analysis. Your task is to analyse an insider threat case study and extract information according to the provided coding schema and Coding Guide.

### Instructions

1. Carefully read the case study and identify information that explicitly appears in the text.

2.The source case ID is: {row['case_id']}
  You MUST use this exact value for: stage1_ingestion.case_id
  Do not generate, infer, or change the case ID.

3. Populate all fields in the provided JSON schema following the definitions and rules in the Coding Guide.

4. Use only the values defined in the Coding Guide for categorical fields. Do not create new categories or modify existing labels.

### Output Requirements

Return only the completed JSON object following the provided schema.

Coding Guide:
{coding_guide_text}

Case Study:
{case_text}
"""
coding_guide_path = "C:/Users/Nadeeka_92/Downloads/Coding_Manual_v2_2_updated.docx"
guide_text = read_word_file(coding_guide_path)

# 3. Updated API call payload to target OpenAI's Chat Completions
def call_gpt(prompt):
    json_schema = generate_expected_json_template()

    # Formulate the payload structure specifically for OpenAI Structured Outputs
    payload = {
        "model": "gpt-4o",  # Using gpt-4o for complex structured outputs
        "messages": [{"role": "user", "content": prompt}],
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "incident_analysis_schema",
                "strict": True,
                "schema": json_schema,
            },
        },
        "temperature": 0,
    }

    headers = {
        "Authorization": "your API key here",
        "Content-Type": "application/json",
    }

    response = requests.post(
        "https://api.openai.com/v1/chat/completions",
        json=payload,
        headers=headers,
    )
    response.raise_for_status()
    print(response)

    
    # Extract the text content, which is a verified JSON string adhering to your schema
    response_string = response.json()["choices"][0]["message"]["content"]

    # Optional: Load it as a native python dictionary before returning
    return json.loads(response_string)

In [18]:
#New Implementation - 12.09.2026
import json
import os
import pandas as pd


# ============================================================
# 1. Load dataset
# ============================================================

csv_path = "cases.csv"

# df = pd.read_csv(csv_path)

# df = pd.read_csv("cases.csv", encoding="cp1252")
df = pd.read_csv("cases.csv", encoding="latin1")


# Folder where individual JSON files will be saved
json_output_folder = "coded_json"

os.makedirs(
    json_output_folder,
    exist_ok=True
)

# List for flattened CSV output
processed_results = []


# ============================================================
# 2. Helper function to flatten JSON
# ============================================================

def flatten_json(nested_dict, prefix=""):

    flattened = {}

    for key, value in nested_dict.items():

        new_key = (
            f"{prefix}_{key}"
            if prefix
            else key
        )

        if isinstance(value, dict):

            flattened.update(
                flatten_json(
                    value,
                    new_key
                )
            )

        elif isinstance(value, list):

            flattened[new_key] = ", ".join(
                map(str, value)
            )

        else:

            flattened[new_key] = value

    return flattened


# ============================================================
# 3. Process each case
# ============================================================

print(
    f"Starting processing for {len(df)} rows..."
)


for index, row in df.iterrows():

    test_text = (
        str(row["description"])
        if pd.notna(row["description"])
        else ""
    )

    if not test_text.strip():

        print(
            f"Skipping row {index}: "
            "Description is empty."
        )

        continue

    print(
        f"Processing row "
        f"{index + 1}/{len(df)}..."
    )

    try:

        # ----------------------------------------------------
        # Build prompt
        # ----------------------------------------------------

        prompt = build_prompt(
            test_text,
            guide_text
        )

        # ----------------------------------------------------
        # Get structured JSON/dictionary from GPT
        # ----------------------------------------------------

        output_dict = call_gpt(prompt)

        # ----------------------------------------------------
        # Determine case ID
        # ----------------------------------------------------

        case_id = (
            output_dict
            .get("stage1_ingestion", {})
            .get("case_id")
        )

        

        # Fallback if GPT did not provide case_id
        if not case_id:

            case_id = f"case_{index + 1:03d}"

        # case_id = str(row["case_id"]).strip()


        # ----------------------------------------------------
        # Save individual JSON file
        # ----------------------------------------------------

        json_output_path = os.path.join(
            json_output_folder,
            f"{case_id}.json"
        )
        
        # json_output_path = os.path.join(
        #     json_output_folder,
        #     f"{case_id}.json"
        # )

        with open(
            json_output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                output_dict,
                f,
                indent=2,
                ensure_ascii=False
            )


        print(
            f"  ✓ JSON saved: "
            f"{json_output_path}"
        )


        # ----------------------------------------------------
        # Flatten JSON for CSV
        # ----------------------------------------------------

        flattened_row = flatten_json(
            output_dict
        )

        flattened_row[
            "original_row_index"
        ] = index

        processed_results.append(
            flattened_row
        )


    except Exception as e:

        print(
            f"❌ Error processing row "
            f"{index}: {e}"
        )

        continue


# ============================================================
# 4. Create combined CSV
# ============================================================

output_df = pd.DataFrame(
    processed_results
)


# ============================================================
# 5. Save CSV
# ============================================================

output_csv_path = "output120926.csv"

output_df.to_csv(
    output_csv_path,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 6. Completion information
# ============================================================

print("\n" + "=" * 60)

print("PROCESSING COMPLETE")

print("=" * 60)

print(
    f"Cases processed: "
    f"{len(processed_results)}"
)

print(
    f"JSON files saved in: "
    f"{json_output_folder}"
)

print(
    f"Combined CSV saved to: "
    f"{output_csv_path}"
)

Starting processing for 10 rows...
Processing row 1/10...
  ✓ JSON saved: coded_json\CS-001.json
Processing row 2/10...
  ✓ JSON saved: coded_json\CS-002.json
Processing row 3/10...
  ✓ JSON saved: coded_json\CS-003.json
Processing row 4/10...
  ✓ JSON saved: coded_json\CS-004.json
Processing row 5/10...
  ✓ JSON saved: coded_json\CS-005.json
Processing row 6/10...
  ✓ JSON saved: coded_json\CS-006.json
Processing row 7/10...
  ✓ JSON saved: coded_json\CS-008.json
Processing row 8/10...
  ✓ JSON saved: coded_json\CS-010.json
Processing row 9/10...
  ✓ JSON saved: coded_json\CS-012.json
Processing row 10/10...
  ✓ JSON saved: coded_json\CS-018.json

PROCESSING COMPLETE
Cases processed: 10
JSON files saved in: coded_json
Combined CSV saved to: output120926.csv
